In [26]:
import pandas as pd
import altair as alt
import numpy as np
import dash
from dash import dcc, html
import dash_deck
import json
import plotly.express as px

In [28]:
df = pd.read_csv("../../data/preprocessed_anime.csv")

# Split the 'Genres' column so that each genre is available.
df_exploded = df.assign(Genres=df['Genres'].str.split(', ')).explode('Genres')

# Calculate the average score for each genre
genre_avg_score = df_exploded.groupby('Genres')['Score'].mean().reset_index()

# Sort the average score in descending order
genre_avg_score = genre_avg_score.sort_values(by='Score', ascending=False)

# Create the bar chart
chart = alt.Chart(genre_avg_score).mark_bar().encode(
    x=alt.X('Score:Q', title='Average Score'),
    y=alt.Y('Genres:N', sort='-x', title='Genre'),
    color=alt.Color('Score:Q', scale=alt.Scale(scheme='blues'), legend=None)
).properties(
    title="Average Score by Genre",
    width=500,
    height=1000
)

# Show chart
chart

alt.Chart(...)

In [30]:
# Convert the chart to a JSON format
chart_json = chart.to_json()

# Initialization
app = dash.Dash(__name__)

# Layout
app.layout = html.Div([
    html.H1("Average Scores for Different Anime Genre", style={'textAlign': 'center'}),
    html.P("This dashboard displays the average scores of different anime genres."),
    
    # Altair Chart
    dcc.Graph(
        id='altair-graph',
        config={'displayModeBar': False},
        figure={'data': [], 'layout': {'height': 1000}}
    ),
    
    # Hidden Div to store the chart JSON
    html.Div(id='chart-json', style={'display': 'none'}, children=chart_json)
])

@app.callback(
    dash.dependencies.Output('altair-graph', 'figure'),
    [dash.dependencies.Input('chart-json', 'children')]
)
def update_chart(chart_json):
    # Load the JSON
    alt_chart_data = json.loads(chart_json)

    # Extract the dataset to handle dynamic keys
    dataset_key = list(alt_chart_data["datasets"].keys())[0] # Get the first dataset key
    df_chart = pd.DataFrame(alt_chart_data["datasets"][dataset_key])  # Convert dataset to DataFrame

    # Convert to Plotly Bar Chart
    fig = px.bar(
        df_chart,
        x="Score",
        y="Genres",
        orientation="h",
        title="Average Score by Genre",
        color="Score",
        color_continuous_scale="blues"
    )

    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height = 1000)
    
    return fig

# Run
if __name__ == '__main__':
    app.run_server(debug=True)